# ⚡ FranchiseOps AI — Milestone 2

---



---


### Enterprise Multi-Agent Franchise Operations Platform


## Step 1 — Install Dependencies


In [ ]:
!pip install -q streamlit pyngrok bcrypt pyjwt pandas numpy scikit-learn joblib transformers accelerate bitsandbytes plotly streamlit-option-menu faker kaggle


## Step 2 — Configure Secrets & Mount Google Drive


In [ ]:
import os

def _get_secret(key):
    """Read from Colab Secrets first, then environment variable."""
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val: return val
    except Exception:
        pass
    return os.environ.get(key, "")

# ── Load all secrets ──────────────────
NGROK_AUTHTOKEN = _get_secret("NGROK_AUTHTOKEN")
HF_TOKEN        = _get_secret("HF_TOKEN")
KAGGLE_API_TOKEN= _get_secret("KAGGLE_API_TOKEN") # <-- Updated for your new KGAT token
EMAIL_PASSWORD  = _get_secret("EMAIL_PASSWORD")
EMAIL_ID        = _get_secret("EMAIL_ID")
JWT_SECRET_KEY  = _get_secret("JWT_SECRET_KEY") or "franchiseops_ai-dev-secret"
ADMIN_EMAIL     = _get_secret("ADMIN_EMAIL_ID") or "infosys@ai"
ADMIN_PASSWORD  = _get_secret("ADMIN_PASSWORD") or "admin@123"

# Expose new Kaggle token credential to the environment
if KAGGLE_API_TOKEN: os.environ["KAGGLE_API_TOKEN"] = KAGGLE_API_TOKEN

# ── Mount Google Drive ─────────────────────────────
try:
    if os.path.exists("/content"):
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        STORAGE_DIR = "/content/drive/MyDrive/FranchiseOps_AI"
        print("✅ Google Drive mounted.")
    else:
        STORAGE_DIR = os.path.abspath("./data/FranchiseOps_AI")
except Exception as e:
    print(f"⚠️  Drive mount skipped ({e}). Using local storage.")
    STORAGE_DIR = os.path.abspath("./data/FranchiseOps_AI")

os.makedirs(STORAGE_DIR, exist_ok=True)
os.makedirs(os.path.join(STORAGE_DIR, "models"), exist_ok=True)
os.makedirs(os.path.join(STORAGE_DIR, "models", "kaggle_cache"), exist_ok=True)
os.makedirs(os.path.join(STORAGE_DIR, "models", "hf_cache"), exist_ok=True)

print(f"\n📁 Storage:  {STORAGE_DIR}")
print(f"🔑 JWT:      {'✅ from Colab Secrets' if _get_secret('JWT_SECRET_KEY') else '⚠️  using dev default'}")
print(f"🔑 Admin:    {ADMIN_EMAIL}")
print(f"🔑 HF_TOKEN: {'✅' if HF_TOKEN else '❌ set in Colab Secrets'}")
print(f"🔑 Kaggle:   {'✅' if KAGGLE_API_TOKEN else '❌ optional — synthetic fallback'}")
print(f"🔑 ngrok:    {'✅' if NGROK_AUTHTOKEN else '❌ set in Colab Secrets'}")
print(f"🔑 Email:    {'✅' if EMAIL_PASSWORD else '❌ optional'}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted.

📁 Storage:  /content/drive/MyDrive/FranchiseOps_AI
🔑 JWT:      ⚠️  using dev default
🔑 Admin:    infosys@ai
🔑 HF_TOKEN: ❌ set in Colab Secrets
🔑 Kaggle:   ✅
🔑 ngrok:    ✅
🔑 Email:    ✅


## Step 3 — Verify GPU & Load Qwen-2.5-3B (4-bit NF4)


In [ ]:
!nvidia-smi


Fri Jul 24 06:12:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   75C    P0             31W /   70W |    2149MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto",
)
print("✅ Qwen-2.5-3B loaded. Footprint (GB):", round(model.get_memory_footprint() / 1e9, 2))


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


✅ Qwen-2.5-3B loaded. Footprint (GB): 2.01


## Step 4 — Write All Application Modules (`llm_engine`, `config`, `auth`, `db`, `agents`, `dashboard`)


In [ ]:
%%writefile llm_engine.py
"""
llm_engine.py — FranchiseOps AI (v4 FINAL — Maximum Speed Edition)
Qwen-2.5-3B-Instruct (4-bit NF4) with:
  • Google Drive Persistent Caching (hf_cache) — instant reload without re-download
  • low_cpu_mem_usage=True + attn_implementation="sdpa" (falls back to "eager") — faster load AND faster generation on T4
  • torch.inference_mode() + use_cache=True + greedy decode — ~1 sec responses
  • Single-Pass generate_debate_and_synthesis() — all 3 agents + synthesis in ~1.5 sec
  • Trimmed max_new_tokens across all 3 generation functions for lower per-call latency
"""
import os, json, re, torch, threading
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from config import HF_TOKEN

MODEL_ID  = "Qwen/Qwen2.5-3B-Instruct"
CACHE_DIR = "/content/drive/MyDrive/FranchiseOps_AI/models/hf_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

_model     = None
_tokenizer = None
_load_lock = threading.Lock()


def get_model():
    global _model, _tokenizer
    if _model is not None:
        return _model, _tokenizer
    with _load_lock:
        if _model is not None:          # someone else finished loading while we waited
            return _model, _tokenizer
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        kw = {"token": HF_TOKEN, "cache_dir": CACHE_DIR} if HF_TOKEN else {"cache_dir": CACHE_DIR}
        _tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, **kw)
        # sdpa (PyTorch's built-in scaled-dot-product-attention kernel) generates
        # noticeably faster than "eager" on T4 -- eager only wins on load time.
        # Fall back to eager automatically if this transformers/torch combo
        # doesn't support sdpa for Qwen2, so this never becomes a new crash.
        try:
            _model = AutoModelForCausalLM.from_pretrained(
                MODEL_ID,
                quantization_config=bnb,
                device_map="auto",
                torch_dtype=torch.float16,
                low_cpu_mem_usage=True,
                attn_implementation="sdpa",
                **kw,
            )
        except Exception:
            _model = AutoModelForCausalLM.from_pretrained(
                MODEL_ID,
                quantization_config=bnb,
                device_map="auto",
                torch_dtype=torch.float16,
                low_cpu_mem_usage=True,
                attn_implementation="eager",
                **kw,
            )
        _model.eval()
    return _model, _tokenizer


def warmup_llm():
    """Load model into GPU memory for instant subsequent generation."""
    try:
        get_model()
        return _model is not None
    except Exception:
        return False


def is_llm_loaded():
    return _model is not None


_warmup_thread_started = False

def start_background_warmup():
    """
    Kicks off model loading in a background thread exactly once per process,
    called at app.py import time. This way the model is already warm -- or
    already warming up -- before anyone opens the AI Copilot tab, instead of
    blocking on someone's first click mid-demo. get_model()'s _load_lock means
    a manual warmup_llm() call or a real chat request made while this thread
    is still loading just waits for it, rather than starting a second,
    duplicate (and GPU-memory-doubling) load.
    """
    global _warmup_thread_started
    if _warmup_thread_started:
        return
    _warmup_thread_started = True
    threading.Thread(target=warmup_llm, daemon=True).start()


def _run(msgs, max_tokens=100, greedy=True):
    """Core low-overhead generation helper."""
    model, tok = get_model()
    tmpl   = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(tmpl, return_tensors="pt").to(model.device)
    gen_kw = dict(
        max_new_tokens=max_tokens,
        use_cache=True,
        pad_token_id=tok.eos_token_id,
        eos_token_id=tok.eos_token_id,
    )
    if greedy:
        gen_kw["do_sample"] = False
    else:
        gen_kw["do_sample"]   = True
        gen_kw["temperature"] = 0.2
        gen_kw["top_p"]       = 0.9
    with torch.inference_mode():
        out = model.generate(**inputs, **gen_kw)
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()


def generate_json(prompt, schema_keys=None):
    """Returns a structured JSON dict from the model — greedy, minimal tokens."""
    sys_p = "You are an AI franchise intelligence engine. Respond ONLY with a valid JSON object."
    if schema_keys:
        sys_p += f" Required keys: {', '.join(schema_keys)}."
    raw = _run(
        [{"role": "system", "content": sys_p}, {"role": "user", "content": prompt}],
        max_tokens=150,
        greedy=True,
    )
    def _repair_json(text):
        text = re.sub(r'```json\s*|\s*```', '', text)
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if m: text = m.group(0)
        # Fix missing commas between key-value pairs (e.g. "val"\n"key": or "val" "key":)
        text = re.sub(r'(["]|\d|true|false)\s*\n\s*(["\w]+":)', r'\1,\n\2', text)
        text = re.sub(r'(["]|\d|true|false)\s+(["\w]+":)', r'\1, \2', text)
        # Fix trailing commas before closing brace
        text = re.sub(r',\s*\}', '}', text)
        return text

    try:
        return json.loads(_repair_json(raw))
    except Exception:
        if schema_keys:
            # Fallback regex extraction of key-value pairs if strict JSON still fails
            out = {}
            for k in schema_keys:
                km = re.search(rf'"{k}"\s*:\s*"([^"]*)"|"{k}"\s*:\s*([^,\}}]+)', raw)
                if km: out[k] = (km.group(1) if km.group(1) is not None else km.group(2)).strip()
                else: out[k] = "N/A"
            if any(v != "N/A" for v in out.values()): return out
        return {"error": "JSON parse failed", "raw": raw}


# ── Agent Roles ───────────────────────────────────────────────────────────────
AGENT_ROLES = {
    "agent1": ("Workforce Retention Agent",
               "You specialise in employee satisfaction, overtime fatigue, and attrition risk."),
    "agent2": ("Outlet Territory Clustering Agent",
               "You specialise in store revenue vs cost clustering, headcount efficiency, tier rating."),
    "agent3": ("Supply Chain & Inventory Advisor Agent",
               "You specialise in weather-driven demand surges, SKU stockout probabilities, lead times."),
}


def generate_debate_and_synthesis(user_query, agent1_context, agent2_context, agent3_context, db_stats=None):
    """
    Single-pass structured generation — outputs Agent 1 / 2 / 3 views + Synthesis.
    Target latency: ~2 sec on T4.
    """
    system_prompt = (
        "You are the FranchiseOps AI Multi-Agent Engine. "
        "Analyze the query and all data. Reply STRICTLY in this format:\n"
        "[AGENT 1]: <1 bullet on workforce/attrition>\n"
        "[AGENT 2]: <1 bullet on outlet clustering/revenue>\n"
        "[AGENT 3]: <1 bullet on inventory/weather>\n"
        "[SYNTHESIS]: <2 sentences executive recommendation>"
    )
    ctx = (
        f"QUERY: {user_query}\n"
        f"A1: {json.dumps(agent1_context)}\n"
        f"A2: {json.dumps(agent2_context)}\n"
        f"A3: {json.dumps(agent3_context)}"
    )
    if db_stats:
        ctx += f"\nDB: {json.dumps(db_stats)}"

    raw = _run(
        [{"role": "system", "content": system_prompt}, {"role": "user", "content": ctx}],
        max_tokens=100,
        greedy=True,
    )
    res = {
        "agent1": "Overtime hours and low satisfaction are primary attrition drivers.",
        "agent2": "Outlet clustering identifies underperforming stores with high cost ratios.",
        "agent3": "Weather-driven demand surges are causing critical SKU stockout risk.",
        "synthesis": raw,
    }
    try:
        for key, tag, nxt in [
            ("agent1", "AGENT 1", "AGENT 2"),
            ("agent2", "AGENT 2", "AGENT 3"),
            ("agent3", "AGENT 3", "SYNTHESIS"),
        ]:
            m = re.search(rf"\[{tag}\]:\s*(.*?)(?=\[{nxt}\]|\Z)", raw, re.DOTALL | re.IGNORECASE)
            if m:
                res[key] = m.group(1).strip()
        m = re.search(r"\[SYNTHESIS\]:\s*(.*)", raw, re.DOTALL | re.IGNORECASE)
        if m:
            res["synthesis"] = m.group(1).strip()
    except Exception:
        pass
    return res


def orchestrate_3_agents_query(user_question, agent1_context, agent2_context, agent3_context, db_stats=None):
    """Fast greedy single-pass answer — target latency ~1.5 sec on T4."""
    sys_p = (
        "You are FranchiseOps AI Orchestrator. "
        "Give a crisp 2-sentence actionable executive answer using all agent data."
    )
    ctx = (
        f"QUERY: {user_question}\n"
        f"A1: {json.dumps(agent1_context)}\n"
        f"A2: {json.dumps(agent2_context)}\n"
        f"A3: {json.dumps(agent3_context)}"
    )
    if db_stats:
        ctx += f"\nDB: {json.dumps(db_stats)}"
    return _run(
        [{"role": "system", "content": sys_p}, {"role": "user", "content": ctx}],
        max_tokens=90,
        greedy=True,
    )


Overwriting llm_engine.py


In [ ]:
%%writefile config.py
import os

def _get_secret(key):
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val: return val
    except Exception:
        pass
    return os.environ.get(key, "")

try:
    from __main__ import (STORAGE_DIR, NGROK_AUTHTOKEN, HF_TOKEN,
                          KAGGLE_API_TOKEN, EMAIL_PASSWORD,
                          ADMIN_EMAIL, ADMIN_PASSWORD, EMAIL_ID)
except ImportError:
    STORAGE_DIR    = ("/content/drive/MyDrive/FranchiseOps_AI"
                      if os.path.exists("/content/drive/MyDrive") else
                      os.path.abspath("./data/FranchiseOps_AI"))
    NGROK_AUTHTOKEN = _get_secret("NGROK_AUTHTOKEN")
    NGROK_AUTH_TOKEN = NGROK_AUTHTOKEN
    HF_TOKEN        = _get_secret("HF_TOKEN")
    KAGGLE_API_TOKEN = _get_secret("KAGGLE_API_TOKEN") # <-- Updated here!
    EMAIL_PASSWORD  = _get_secret("EMAIL_PASSWORD")
    EMAIL_ID        = _get_secret("EMAIL_ID")
    JWT_SECRET_KEY  = _get_secret("JWT_SECRET_KEY") or "franchiseops-dev-secret-changeme"
    ADMIN_EMAIL     = _get_secret("ADMIN_EMAIL_ID")  or "infosys@ai"
    ADMIN_PASSWORD  = _get_secret("ADMIN_PASSWORD")  or "admin@123"

os.makedirs(STORAGE_DIR, exist_ok=True)
DB_PATH          = os.path.join(STORAGE_DIR, "franchiseops.db")
MODELS_DIR       = os.path.join(STORAGE_DIR, "models")
KAGGLE_CACHE_DIR = os.path.join(MODELS_DIR, "kaggle_cache")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(KAGGLE_CACHE_DIR, exist_ok=True)

AGENT1_MODEL_PATH = os.path.join(MODELS_DIR, "attrition_lr.joblib")
KMEANS_MODEL_PATH = os.path.join(MODELS_DIR, "kmeans_outlets.joblib")
AGENT2_MODEL_PATH = KMEANS_MODEL_PATH
AGENT2_REG_PATH   = os.path.join(MODELS_DIR, "revenue_rf.joblib")
AGENT3_MODEL_PATH = os.path.join(MODELS_DIR, "inventory_demand_gb.joblib")

Overwriting config.py


In [ ]:
%%writefile ui_theme.py
"""
Shared ui_theme.py for FreightQuote AI & FranchiseOps AI
Exact Neo-Brutalist UI styling, layout cards, and status badges.
"""
import streamlit as st

COLORS = {
    "bg_main":       "#fffffe",
    "bg_card":       "#fffffe",
    "bg_alt":        "#f2f4f6",
    "text_heading":  "#272343",
    "text_body":     "#2d334a",
    "text_main":     "#2d334a",
    "text_muted":    "#626880",
    "border":        "#272343",
    "accent":        "#ffd803",
    "accent_subtle": "#ffe866",
    "accent_text":   "#272343",
    "cyan":          "#e3f6f5",
    "pink":          "#ffd3e2",
    "green":         "#34d399",
    "yellow":        "#fbbf24",
    "red":           "#f87171",
}

NEO_BRUTALIST_CSS = f"""
<style>
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&family=Space+Grotesk:wght@600;700&family=JetBrains+Mono:wght@500;700&display=swap');

html, body, [class*="css"] {{
    font-family: 'Plus Jakarta Sans', sans-serif;
    color: {COLORS["text_body"]};
    background-color: {COLORS["bg_main"]};
}}

h1, h2, h3, h4, h5, h6 {{
    font-family: 'Space Grotesk', sans-serif;
    color: {COLORS["text_heading"]};
    font-weight: 700;
}}

.pn-card {{
    background: {COLORS["bg_card"]};
    border: 3px solid {COLORS["border"]};
    border-radius: 12px;
    padding: 20px;
    margin-bottom: 20px;
    box-shadow: 6px 6px 0px {COLORS["border"]};
    transition: transform 0.15s ease, box-shadow 0.15s ease;
}}
.pn-card:hover {{
    transform: translate(-2px, -2px);
    box-shadow: 8px 8px 0px {COLORS["border"]};
}}
.pn-card-alt {{
    background: {COLORS["cyan"]};
    border: 3px solid {COLORS["border"]};
    border-radius: 12px;
    padding: 20px;
    margin-bottom: 20px;
    box-shadow: 6px 6px 0px {COLORS["border"]};
}}

.pn-badge {{
    display: inline-block;
    padding: 4px 12px;
    border: 2px solid {COLORS["border"]};
    border-radius: 6px;
    font-family: 'JetBrains Mono', monospace;
    font-weight: 700;
    font-size: 13px;
    box-shadow: 2px 2px 0px {COLORS["border"]};
    text-transform: uppercase;
}}
.agent-badge {{
    display: inline-block;
    padding: 4px 14px;
    background: {COLORS["accent"]};
    color: {COLORS["text_heading"]};
    border: 2px solid {COLORS["border"]};
    border-radius: 8px;
    font-family: 'Space Grotesk', sans-serif;
    font-weight: 700;
    font-size: 14px;
    box-shadow: 3px 3px 0px {COLORS["border"]};
}}

/* Streamlit Buttons Matching Login Portal */
div.stButton > button {{
    background: #ffd803 !important;
    color: #272343 !important;
    font-family: 'Space Grotesk', sans-serif !important;
    font-weight: 700 !important;
    border: 3px solid #272343 !important;
    border-radius: 10px !important;
    padding: 10px 22px !important;
    box-shadow: 4px 4px 0px #272343 !important;
    transition: all 0.15s ease !important;
}}
div.stButton > button:hover {{
    transform: translate(-2px, -2px) !important;
    box-shadow: 6px 6px 0px #272343 !important;
    background: #ffe866 !important;
}}

/* Streamlit Inputs & Selectboxes Matching Login Portal */
div[data-baseweb="input"] > div, div[data-baseweb="select"] > div {{
    background: #fffffe !important;
    border: 3px solid #272343 !important;
    border-radius: 8px !important;
    box-shadow: 3px 3px 0px #272343 !important;
}}

/* Streamlit Tabs Matching Login Portal */
button[data-baseweb="tab"] {{
    font-family: 'Space Grotesk', sans-serif !important;
    font-weight: 700 !important;
    color: #2d334a !important;
}}
button[data-baseweb="tab"][aria-selected="true"] {{
    color: #272343 !important;
    border-bottom: 3px solid #ffd803 !important;
}}
</style>
"""

def inject_css():
    st.markdown(NEO_BRUTALIST_CSS, unsafe_allow_html=True)

def apply_theme():
    inject_css()

def render_header(title, subtitle="", icon="⚡"):
    inject_css()
    st.markdown(f"""
    <div style="background:{COLORS['bg_card']};border:3px solid {COLORS['border']};border-radius:14px;padding:22px 28px;margin-bottom:24px;box-shadow:6px 6px 0px {COLORS['border']};">
        <div style="display:flex;align-items:center;gap:16px;">
            <div style="font-size:42px;line-height:1;">{icon}</div>
            <div>
                <h1 style="margin:0;font-size:26px;letter-spacing:-0.5px;">{title}</h1>
                <p style="margin:4px 0 0;color:{COLORS['text_muted']};font-size:14px;">{subtitle}</p>
            </div>
        </div>
    </div>
    """, unsafe_allow_html=True)

def render_card(content, alt=False):
    c_class = "pn-card-alt" if alt else "pn-card"
    st.markdown(f'<div class="{c_class}">{content}</div>', unsafe_allow_html=True)

def risk_badge(text, level="Low"):
    color_map = {"Low": COLORS["green"], "Medium": COLORS["yellow"], "High": COLORS["red"], "Critical": COLORS["red"]}
    c = color_map.get(level, COLORS["cyan"])
    return f'<span class="pn-badge" style="background:{c};">{text}</span>'


Overwriting ui_theme.py


In [ ]:
%%writefile auth.py
import sqlite3, jwt, bcrypt, datetime, time, streamlit as st
try:
    from config import DB_PATH, JWT_SECRET_KEY
    JWT_SECRET = JWT_SECRET_KEY
except (ImportError, AttributeError):
    from config import DB_PATH
    JWT_SECRET = "super-secret-franchiseops-key-2026"
from ui_theme import COLORS

def get_conn():
    return sqlite3.connect(DB_PATH, check_same_thread=False)

def hash_txt(t):
    return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()

def check_txt(t, h):
    try: return bcrypt.checkpw(t.encode(), h.encode()) if h else False
    except: return False

def make_jwt(email, username):
    return jwt.encode({"email": email, "username": username, "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=6)}, JWT_SECRET, algorithm="HS256")

def check_password_strength(pw):
    if len(pw) < 5: return "Weak"
    if len(pw) < 10: return "Average"
    return "Good"

@st.cache_resource
def init_auth():
    with get_conn() as conn:
        if not conn.execute("SELECT id FROM users WHERE email='infosys@ai'").fetchone():
            # Seed default admin account
            conn.execute("""INSERT OR IGNORE INTO users
                         (username, email, password_hash, security_question, security_answer_hash, role, account_status)
                         VALUES (?, ?, ?, ?, ?, ?, ?)""",
                         ("Administrator", "infosys@ai", hash_txt("admin@123"), "What is your pet name?", hash_txt("admin"), "Admin", "active"))
            conn.commit()

def render_auth_portal():
    init_auth()
    if "token" not in st.session_state: st.session_state["token"] = None
    if "otp_resend_count" not in st.session_state: st.session_state["otp_resend_count"] = 0
    if "otp_next_allowed" not in st.session_state: st.session_state["otp_next_allowed"] = 0.0

    st.markdown(f"""
    <div style="text-align:center;padding:1.5rem 0 1rem;">
        <div style="font-size:44px;margin-bottom:8px;">⚡</div>
        <h1 style="font-size:2rem !important;margin:0;">FranchiseOps AI Portal</h1>
        <p style="color:{COLORS['text_muted']};font-size:14px;margin:4px 0 0;">Enterprise Multi-Agent Franchise Intelligence System</p>
    </div>
    """, unsafe_allow_html=True)

    c1, c2, c3 = st.columns([1, 2, 1])
    with c2:
        tab1, tab2, tab3 = st.tabs(["🔐 Sign In", "📝 Register Account", "🔑 Reset Password"])

        with tab1:
            login_email = st.text_input("Email / Username", key="l_email", placeholder="infosys@ai")
            login_pw = st.text_input("Password", type="password", key="l_pw", placeholder="••••••••")
            if st.button("🚀 Sign In to Portal", key="btn_login"):
                with get_conn() as conn:
                    user = conn.execute("SELECT id, username, email, password_hash, role, failed_attempts, lock_until, account_status FROM users WHERE email=? OR username=?", (login_email, login_email)).fetchone()

                if user:
                    u_id, u_name, u_email, u_hash, u_role, u_fails, u_lock, u_status = user
                    now = datetime.datetime.now()

                    if u_status == 'locked':
                        st.error("❌ Account permanently locked due to 5 failed attempts. Only the System Administrator can unlock this account via the Admin Dashboard.")
                    elif u_lock and now < datetime.datetime.strptime(u_lock, '%Y-%m-%d %H:%M:%S.%f'):
                        time_left = (datetime.datetime.strptime(u_lock, '%Y-%m-%d %H:%M:%S.%f') - now).seconds // 60
                        st.error(f"Account temporarily locked for {time_left+1} minutes due to failed attempts.")
                    elif check_txt(login_pw, u_hash):
                        with get_conn() as conn:
                            conn.execute("UPDATE users SET failed_attempts=0, lock_until=NULL WHERE id=?", (u_id,))
                            conn.commit()
                        st.session_state["token"] = make_jwt(u_email, u_name)
                        st.session_state["username"] = u_name
                        st.session_state["role"] = u_role
                        st.success(f"Welcome back, {u_name} [{u_role}]!")
                        st.rerun()
                    else:
                        u_fails += 1
                        lock_time = None
                        if u_fails == 3:
                            lock_time = now + datetime.timedelta(minutes=5)
                            st.warning("Account temporarily locked for 5 minutes due to 3 failed attempts.")
                        elif u_fails == 4:
                            lock_time = now + datetime.timedelta(minutes=15)
                            st.warning("Account temporarily locked for 15 minutes due to 4 failed attempts.")
                        elif u_fails >= 5:
                            st.error("❌ Account permanently locked due to 5 failed attempts. Only the System Administrator can unlock this account via the Admin Dashboard.")
                        else:
                            st.error("Invalid password.")

                        with get_conn() as conn:
                            if u_fails >= 5:
                                conn.execute("UPDATE users SET failed_attempts=?, account_status='locked', lock_until=NULL WHERE id=?", (u_fails, u_id))
                            else:
                                conn.execute("UPDATE users SET failed_attempts=?, lock_until=? WHERE id=?", (u_fails, lock_time, u_id))
                            conn.commit()
                else:
                    st.error("Invalid email/username or password.")

        with tab2:
            r_user = st.text_input("Username", key="r_u")
            r_email = st.text_input("Email Address", key="r_e")
            r_pw = st.text_input("Create Password", type="password", key="r_p")
            r_role = st.selectbox("Select Enterprise Role", ["Franchise Owner", "Regional Operations Manager", "Store Manager", "Supply Chain Analyst"], key="r_role")
            r_q = st.selectbox("Security Question", ["What is your pet name?", "What city were you born in?", "What is your favorite school teacher's name?"], key="r_q")
            r_a = st.text_input("Security Answer", key="r_a")
            if st.button("✨ Create Franchisee Account", key="btn_reg"):
                if r_user and r_email and r_pw and r_a:
                    strength = check_password_strength(r_pw)
                    if strength == "Weak":
                        st.warning("Password too weak (minimum 5 characters required).")
                    else:
                        if strength == "Average":
                            st.info("Average strength (10+ characters recommended for enterprise security).")
                        else:
                            st.success("Good password strength - proceed with bcrypt hashing.")
                        try:
                            with get_conn() as conn:
                                conn.execute("INSERT INTO users (username, email, password_hash, security_question, security_answer_hash, role) VALUES (?, ?, ?, ?, ?, ?)",
                                             (r_user, r_email, hash_txt(r_pw), r_q, hash_txt(r_a.lower().strip()), r_role))
                                conn.commit()
                            st.success(f"Account registered with role [{r_role}]! Please switch to Sign In tab.")
                        except Exception as e:
                            st.error(f"Registration failed: Email or username may already exist.")
                else:
                    st.warning("Please fill out all fields.")

        with tab3:
            f_email = st.text_input("Registered Email", key="f_e")
            if st.button("Request OTP / Reset Code", key="btn_f1"):
                current_time = time.time()
                if current_time < st.session_state["otp_next_allowed"]:
                    wait_sec = int(st.session_state["otp_next_allowed"] - current_time)
                    if wait_sec > 300: st.error("Too many OTP requests. Please wait 1 hour before trying again.")
                    elif wait_sec > 180: st.warning(f"Please wait 5 minutes before requesting another OTP.")
                    elif wait_sec > 60: st.warning(f"Please wait 3 minutes before requesting another OTP.")
                    else: st.warning(f"Please wait 60 seconds before requesting another OTP.")
                else:
                    st.session_state["otp_resend_count"] += 1
                    count = st.session_state["otp_resend_count"]
                    if count == 1: cooldown = 60
                    elif count == 2: cooldown = 180
                    elif count == 3: cooldown = 300
                    else: cooldown = 3600

                    st.session_state["otp_next_allowed"] = current_time + cooldown

                    with get_conn() as conn:
                        u = conn.execute("SELECT security_question FROM users WHERE email=?", (f_email,)).fetchone()
                    if u:
                        st.session_state["reset_email"] = f_email
                        st.session_state["reset_q"] = u[0]
                        st.success("OTP / Verification processed! Answer your security question below.")
                    else:
                        st.error("Email not found.")

            if st.session_state.get("reset_email"):
                st.info(f"Security Question: **{st.session_state.get('reset_q')}**")
                ans_try = st.text_input("Enter Answer", key="f_ans")
                new_pw = st.text_input("New Password", type="password", key="f_npw")
                if st.button("Confirm Password Reset", key="btn_f2"):
                    strength = check_password_strength(new_pw)
                    if strength == "Weak":
                        st.warning("Password too weak (minimum 5 characters required).")
                    else:
                        with get_conn() as conn:
                            u_hash = conn.execute("SELECT security_answer_hash FROM users WHERE email=?", (st.session_state["reset_email"],)).fetchone()
                        if u_hash and check_txt(ans_try.lower().strip(), u_hash[0]):
                            with get_conn() as conn:
                                conn.execute("UPDATE users SET password_hash=?, failed_attempts=0, lock_until=NULL, account_status='active' WHERE email=?", (hash_txt(new_pw), st.session_state["reset_email"]))
                                conn.commit()
                            if strength == "Average": st.info("Average strength (10+ characters recommended for enterprise security).")
                            else: st.success("Good password strength - proceed with bcrypt hashing.")
                            st.success("Password reset successfully! Account unlocked. Please sign in.")
                            st.session_state["reset_email"] = None
                        else:
                            st.error("Incorrect security answer.")

Overwriting auth.py


In [ ]:
%%writefile db.py
import sqlite3
from config import DB_PATH

def get_conn():
    return sqlite3.connect(DB_PATH, check_same_thread=False)

def init_db():
    with get_conn() as conn:
        conn.execute("""CREATE TABLE IF NOT EXISTS outlets (
            outlet_id TEXT PRIMARY KEY, outlet_name TEXT, city TEXT,
            monthly_revenue REAL, monthly_costs REAL, staff_headcount INTEGER,
            avg_overtime_hours REAL, customer_satisfaction REAL,
            tier_cluster TEXT, attrition_risk_level TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.execute("""CREATE TABLE IF NOT EXISTS staff (
            staff_id TEXT PRIMARY KEY, outlet_id TEXT, employee_name TEXT,
            role TEXT, monthly_salary REAL, weekly_overtime_hrs REAL,
            job_satisfaction INTEGER, employee_age INTEGER, tenure_years REAL,
            work_life_balance INTEGER, predicted_attrition_prob REAL,
            intervention_status TEXT DEFAULT 'Active')""")
        conn.execute("""CREATE TABLE IF NOT EXISTS inventory_records (
            record_id INTEGER PRIMARY KEY AUTOINCREMENT, outlet_id TEXT,
            sku_name TEXT, current_stock INTEGER, weekly_demand INTEGER,
            reorder_threshold INTEGER, stockout_risk_prob REAL,
            last_updated TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.execute("""CREATE TABLE IF NOT EXISTS merged_datasets (
            id INTEGER PRIMARY KEY AUTOINCREMENT, agent_target TEXT, dataset_source TEXT,
            outlet_id TEXT, employee_age INTEGER, overtime_hours REAL,
            job_satisfaction INTEGER, attrition_target INTEGER, monthly_sales_usd REAL,
            operating_cost_usd REAL, tier_cluster_label INTEGER, sku_demand INTEGER,
            weather_impact_factor REAL, stockout_target INTEGER,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")

        # Update users table with progressive lockout columns matching Milestone 2 spec
        conn.execute("""CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE,
            email TEXT UNIQUE, password_hash TEXT,
            security_question TEXT, security_answer_hash TEXT,
            role TEXT DEFAULT 'User',
            failed_attempts INTEGER DEFAULT 0,
            lock_until TIMESTAMP DEFAULT NULL,
            account_status TEXT DEFAULT 'active',
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")

        # Alter tables to catch older schemas
        try: conn.execute("ALTER TABLE users ADD COLUMN security_question TEXT")
        except Exception: pass
        try: conn.execute("ALTER TABLE users ADD COLUMN security_answer_hash TEXT")
        except Exception: pass
        try: conn.execute("ALTER TABLE users ADD COLUMN failed_attempts INTEGER DEFAULT 0")
        except Exception: pass
        try: conn.execute("ALTER TABLE users ADD COLUMN lock_until TIMESTAMP DEFAULT NULL")
        except Exception: pass
        try: conn.execute("ALTER TABLE users ADD COLUMN account_status TEXT DEFAULT 'active'")
        except Exception: pass

        conn.execute("""CREATE TABLE IF NOT EXISTS ml_models (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            agent_name TEXT, model_name TEXT, r2_score REAL,
            rmse REAL, accuracy REAL, training_rows INTEGER,
            file_path TEXT, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.execute("""CREATE TABLE IF NOT EXISTS notifications (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            channel TEXT, recipient TEXT, subject TEXT, message TEXT,
            status TEXT DEFAULT 'Sent',
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.execute("""CREATE TABLE IF NOT EXISTS chat_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT NOT NULL, role TEXT NOT NULL, content TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.commit()

def save_ml_metrics(agent_name, model_name, r2, rmse, acc, rows, path):
    with get_conn() as conn:
        conn.execute("INSERT INTO ml_models "
                     "(agent_name,model_name,r2_score,rmse,accuracy,training_rows,file_path) "
                     "VALUES (?,?,?,?,?,?,?)",
                     (agent_name, model_name, r2, rmse, acc, rows, path))
        conn.commit()

def load_chat_history(username, conn_fn=None, limit=60):
    fn = conn_fn or get_conn
    with fn() as conn:
        rows = conn.execute(
            "SELECT role,content FROM chat_history WHERE username=? "
            "ORDER BY id DESC LIMIT ?", (username, limit)).fetchall()
    return [{"role":r[0],"content":r[1]} for r in reversed(rows)]

def save_chat_message(username, role, content, conn_fn=None):
    fn = conn_fn or get_conn
    with fn() as conn:
        conn.execute("INSERT INTO chat_history (username,role,content) VALUES (?,?,?)",
                     (username, role, content))
        conn.commit()

def clear_chat_history(username, conn_fn=None):
    fn = conn_fn or get_conn
    with fn() as conn:
        conn.execute("DELETE FROM chat_history WHERE username=?", (username,))
        conn.commit()

Overwriting db.py


In [ ]:
%%writefile weather_context.py
"""
weather_context.py for FranchiseOps AI
Simulates local Indian city weather disruptions and logistics delays across franchise outlets.
"""
import random

CITY_WEATHER_REPORTS = {
    "Mumbai (MH)": {"status": "Heavy Monsoon Rain & Waterlogging", "temp_c": 28, "demand_impact_pct": -18.0, "supply_delay_days": 2, "attrition_stress": "High"},
    "Bengaluru (KA)": {"status": "Pleasant / Light Showers", "temp_c": 24, "demand_impact_pct": 12.0, "supply_delay_days": 0, "attrition_stress": "Normal"},
    "Delhi NCR (DL)": {"status": "Intense Summer Heatwave & Smog", "temp_c": 42, "demand_impact_pct": 15.0, "supply_delay_days": 1, "attrition_stress": "High"},
    "Hyderabad (TG)": {"status": "Clear & Warm", "temp_c": 33, "demand_impact_pct": 8.0, "supply_delay_days": 0, "attrition_stress": "Normal"},
    "Chennai (TN)": {"status": "Humid & Coastal Showers", "temp_c": 35, "demand_impact_pct": -5.0, "supply_delay_days": 1, "attrition_stress": "Medium"},
    "Pune (MH)": {"status": "Cloudy & Breezy", "temp_c": 26, "demand_impact_pct": 10.0, "supply_delay_days": 0, "attrition_stress": "Normal"},
    "Ahmedabad (GJ)": {"status": "Dry & High Heat", "temp_c": 40, "demand_impact_pct": -8.0, "supply_delay_days": 1, "attrition_stress": "Medium"},
    "Kolkata (WB)": {"status": "Thunderstorms & High Humidity", "temp_c": 32, "demand_impact_pct": -12.0, "supply_delay_days": 2, "attrition_stress": "High"}
}

def get_city_weather(city_name):
    for k, v in CITY_WEATHER_REPORTS.items():
        if k.lower() in city_name.lower() or city_name.lower() in k.lower():
            return {"city": k, **v}
    return {"city": city_name, "status": "Fair Weather Conditions", "temp_c": 30, "demand_impact_pct": 0.0, "supply_delay_days": 0, "attrition_stress": "Normal"}

def get_weather_report(port_name):
    return {"port": port_name, "status": "Normal Marine Conditions", "temp_c": 25, "wind_kt": 15, "delay_penalty_multiplier": 1.00}


Overwriting weather_context.py


In [ ]:
%%writefile notifications.py
"""
FranchiseOps AI - notifications.py
Multi-channel alert center simulating SMS, Email, and In-App notifications stored in SQLite.
"""
from db import get_conn

def send_alert(channel, recipient, subject, message):
    with get_conn() as conn:
        conn.execute("INSERT INTO notifications (channel, recipient, subject, message, status) VALUES (?, ?, ?, ?, ?)",
                     (channel, recipient, subject, message, "Delivered"))
        conn.commit()
    print(f"[{channel.upper()}] To: {recipient} | Subject: {subject} | Status: Delivered")

def get_recent_alerts(limit=15):
    with get_conn() as conn:
        return conn.execute("SELECT id, channel, recipient, subject, message, created_at FROM notifications ORDER BY id DESC LIMIT ?", (limit,)).fetchall()


Overwriting notifications.py


In [ ]:
%%writefile seed_data.py
"""
FranchiseOps AI - seed_data.py
Pre-seeds the database with realistic outlets, staff members, shift logs, and inventory benchmarks.
"""
from db import get_conn, init_db
from notifications import send_alert

def seed_all():
    init_db()
    with get_conn() as conn:
        # Seed Outlets
        if not conn.execute("SELECT count(*) FROM outlets").fetchone()[0]:
            outlets = [
                ("OUT-101", "Mumbai Flagship Store", "Mumbai (MH)", 145000, 112000, 24, 18.5, 4.2, "Tier 3 (At-Risk)", "High Attrition"),
                ("OUT-102", "Bengaluru Tech Hub Cafe", "Bengaluru (KA)", 285000, 165000, 32, 4.2, 4.8, "Tier 1 (Apex)", "Low Attrition"),
                ("OUT-103", "Delhi NCR Metro Express", "Delhi NCR (DL)", 210000, 155000, 28, 14.0, 4.5, "Tier 2 (Stable)", "Moderate Attrition"),
                ("OUT-104", "Hyderabad Central Hub", "Hyderabad (TG)", 125000, 118000, 18, 22.0, 3.8, "Tier 3 (At-Risk)", "Critical Attrition"),
                ("OUT-105", "Chennai Coastal Kiosk", "Chennai (TN)", 195000, 138000, 26, 6.5, 4.7, "Tier 1 (Apex)", "Low Attrition"),
                ("OUT-106", "Pune IT Park Outlet", "Pune (MH)", 172000, 129000, 22, 9.8, 4.4, "Tier 2 (Stable)", "Low Attrition"),
            ]
            conn.executemany("INSERT INTO outlets VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, CURRENT_TIMESTAMP)", outlets)

        # Seed Staff
        if not conn.execute("SELECT count(*) FROM staff").fetchone()[0]:
            staff = [
                ("ST-5001", "OUT-101", "Marcus Vance", "Shift Supervisor", 3920.0, 21.0, 2, 32, 4.5, 2, 0.82, "Retention Bonus Offered"),
                ("ST-5002", "OUT-101", "Elena Rostova", "Barista / Cashier", 2880.0, 19.5, 2, 26, 2.0, 2, 0.79, "Schedule Adjusted"),
                ("ST-5003", "OUT-102", "David Chen", "Store Manager", 5120.0, 3.5, 5, 41, 8.5, 4, 0.12, "Stable"),
                ("ST-5004", "OUT-104", "Samantha Diaz", "Kitchen Lead", 3360.0, 24.5, 1, 29, 3.0, 1, 0.89, "Immediate Review Required"),
                ("ST-5005", "OUT-105", "James Wilson", "Team Lead", 4000.0, 5.0, 4, 36, 6.0, 3, 0.18, "Stable"),
            ]
            conn.executemany("INSERT INTO staff (staff_id, outlet_id, employee_name, role, monthly_salary, weekly_overtime_hrs, job_satisfaction, employee_age, tenure_years, work_life_balance, predicted_attrition_prob, intervention_status) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)", staff)

        # Seed Inventory
        if not conn.execute("SELECT count(*) FROM inventory_records").fetchone()[0]:
            inventory = [
                ("OUT-101", "Premium Coffee Beans (Kg)", 140, 320, 180, 0.84),
                ("OUT-101", "Organic Milk Syrups (L)", 85, 190, 100, 0.78),
                ("OUT-102", "Premium Coffee Beans (Kg)", 580, 450, 250, 0.12),
                ("OUT-104", "Eco-Packaging Cups (Box)", 40, 210, 150, 0.91),
                ("OUT-105", "Artisan Tea Blends (Kg)", 310, 220, 140, 0.15),
            ]
            conn.executemany("INSERT INTO inventory_records (outlet_id, sku_name, current_stock, weekly_demand, reorder_threshold, stockout_risk_prob) VALUES (?, ?, ?, ?, ?, ?)", inventory)
            conn.commit()

    send_alert("Email", "franchisee@franchiseops.ai", "Franchise Operations Initialized", "Database seeded with 6 regional outlets, staff logs, and inventory benchmarks.")
    print("✅ Database pre-seeded successfully.")


Overwriting seed_data.py


In [ ]:
%%writefile admin_dash.py
import subprocess, datetime
import streamlit as st
import pandas as pd
import plotly.express as px
from db import get_conn
from auth import hash_txt, check_password_strength
from notifications import get_recent_alerts
from ui_theme import render_card, COLORS

_APP_START = datetime.datetime.now()

def _smi(query):
    try:
        r = subprocess.run(["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=3)
        return r.stdout.strip()
    except: return "N/A"

def render_admin_dashboard(project="franchise"):
    render_card('<h3 style="margin:0;">🛡️ System Administration</h3>')

    tab_users, tab_models, tab_health = st.tabs(["👥 User Lifecycle Management", "📈 ML Model Card", "⚙️ System Health"])

    with tab_users:
        render_card('<h4 style="margin:0 0 10px;">Add New User</h4>')
        with st.form('add_user_form', clear_on_submit=True):
            col1, col2 = st.columns(2)
            n_usr = col1.text_input("Username")
            n_email = col2.text_input("Email")
            n_pw = col1.text_input("Initial Password", type="password")
            n_role = col2.selectbox("Role", ["Admin", "Franchise Owner", "Store Manager", "Supply Chain Analyst"])
            if st.form_submit_button("➕ Create Account"):
                if n_usr and n_email and n_pw:
                    if check_password_strength(n_pw) == "Weak":
                        st.error("Password too weak (minimum 5 characters required).")
                    else:
                        try:
                            with get_conn() as conn:
                                conn.execute("INSERT INTO users (username, email, password_hash, role) VALUES (?, ?, ?, ?)",
                                             (n_usr, n_email, hash_txt(n_pw), n_role))
                                conn.commit()
                            st.success(f"User {n_usr} created successfully as {n_role}!")
                            st.rerun()
                        except Exception as e:
                            st.error("Failed to create user (Email/Username might exist).")
                else:
                    st.warning("All fields are required.")

        st.markdown("---")
        render_card('<h4 style="margin:0 0 10px;">Manage Existing Users</h4>')
        with get_conn() as conn:
            users_df = pd.read_sql("SELECT id, username, role, email, failed_attempts, account_status FROM users ORDER BY id DESC", conn)

        for _, row in users_df.iterrows():
            uc1, uc2, uc3, uc4, uc5 = st.columns([2, 2, 2, 2, 1])
            uc1.markdown(f"**{row['username']}**")
            uc2.markdown(f"[{row['role']}]")
            uc3.markdown(f"Fails: {row['failed_attempts']}")

            with uc4:
                if row['account_status'] == 'locked' or row['failed_attempts'] >= 3:
                    if st.button("🔓 Unlock Account", key=f"unlock_{row['id']}"):
                        with get_conn() as c:
                            c.execute("UPDATE users SET failed_attempts=0, lock_until=NULL, account_status='active' WHERE id=?", (row["id"],))
                        st.success("User account unlocked successfully.")
                        st.rerun()
                else:
                    st.markdown("<span style='color:green;'>Active</span>", unsafe_allow_html=True)
            with uc5:
                if st.button("🗑️", key=f"del_user_{row['id']}", help="Delete User"):
                    with get_conn() as c:
                        c.execute("DELETE FROM users WHERE id=?", (row["id"],))
                    st.success(f"Deleted {row['username']}")
                    st.rerun()

    with tab_models:
        render_card('<h4 style="margin:0 0 10px;">Training Transparency Metrics</h4>')
        with get_conn() as conn:
            try:
                ml_df = pd.read_sql("SELECT agent_name, model_name, r2_score as metric_score, accuracy, training_rows, created_at FROM ml_models ORDER BY id DESC", conn)
                st.dataframe(ml_df, use_container_width=True, hide_index=True)
            except Exception:
                st.info("No model training records found.")

    with tab_health:
        h1, h2, h3 = st.columns(3)
        h1.metric("GPU VRAM Used", _smi("memory.used"))
        h2.metric("GPU Utilization", _smi("utilization.gpu") + "%")
        h3.metric("App Uptime", str(datetime.datetime.now() - _APP_START).split(".")[0])

Overwriting admin_dash.py


In [ ]:
%%writefile agent2_franchise.py
"""
agent2_franchise.py — Enriched Agent 2: Outlet Territory Clustering & City Weather
New features: City demand surge chart, revenue vs weather scatter, AI territory advisory.
Extended Indian cities + global franchise locations.
"""
import pandas as pd
import streamlit as st
import plotly.express as px
from ui_theme import render_card, COLORS
from db import get_conn
from weather_context import get_city_weather
from llm_engine import orchestrate_3_agents_query

# ── Full outlet / city list (heavy India coverage) ───────────────────────────
INDIA_CITIES = [
    "Mumbai (MH)", "Delhi (DL)", "Bengaluru (KA)", "Hyderabad (TS)",
    "Chennai (TN)", "Pune (MH)", "Kolkata (WB)", "Ahmedabad (GJ)",
    "Jaipur (RJ)", "Surat (GJ)", "Lucknow (UP)", "Chandigarh (PB)",
    "Bhopal (MP)", "Indore (MP)", "Nagpur (MH)", "Coimbatore (TN)",
    "Kochi (KL)", "Visakhapatnam (AP)", "Patna (BR)", "Ranchi (JH)",
]
GLOBAL_CITIES = [
    "Chicago (IL)", "Los Angeles (CA)", "New York (NY)", "Houston (TX)",
    "London (UK)", "Dubai (AE)", "Singapore (SG)",
]
ALL_CITIES = INDIA_CITIES + GLOBAL_CITIES


def render_agent2_franchise(agent2_c, agent2_r, username, db_stats, a1_ctx, a3_ctx,
                             send_alert, confidence_band):
    render_card('<h3 style="margin:0;">🏬 Agent 2: Outlet Territory Clustering</h3>')

    with get_conn() as conn:
        try:
            out_df = pd.read_sql("SELECT * FROM outlets", conn)
        except Exception:
            out_df = pd.DataFrame()

    c1, c2 = st.columns([1.3, 1])
    with c1:
        if not out_df.empty:
            st.dataframe(
                out_df[["outlet_id", "outlet_name", "city",
                        "monthly_revenue", "monthly_costs", "tier_cluster"]],
                use_container_width=True, hide_index=True)
            fig = px.scatter(
                out_df, x="monthly_costs", y="monthly_revenue",
                color="tier_cluster", size="staff_headcount",
                hover_name="outlet_name",
                title="Revenue vs Cost Clustering",
                color_discrete_sequence=["#34d399", "#ffd803", "#f87171"])
            fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                              height=280, margin=dict(l=10, r=10, t=40, b=10))
            st.plotly_chart(fig, use_container_width=True)

    with c2:
        render_card('<h4 style="margin:0 0 10px;">Simulate New Outlet</h4>')
        city_sel = st.selectbox("City", ALL_CITIES)
        new_rev  = st.number_input("Monthly Revenue (₹)", 80000.0, 2000000.0, 380000.0, step=10000.0)
        new_cost = st.number_input("Monthly Costs (₹)", 50000.0, 1500000.0, 260000.0, step=10000.0)
        new_hc   = st.slider("Staff Headcount", 5, 80, 22)
        if st.button("⚡ Predict Tier Cluster", key="btn_predict_tier"):
            idx = (agent2_c.predict([[new_rev, new_cost, new_hc]])[0]
                   if agent2_c else (0 if new_rev > 500000 else (2 if (new_rev - new_cost) < 40000 else 1)))
            tiers = ["Tier 1 (Apex)", "Tier 2 (Stable)", "Tier 3 (At-Risk)"]
            cols  = ["#34d399", "#ffd803", "#f87171"]
            st.markdown(
                f'<div style="background:{cols[idx % 3]};padding:14px;border-radius:12px;'
                f'border:2px solid #272343;font-weight:700;font-size:16px;">'
                f'{tiers[idx % 3]}</div>', unsafe_allow_html=True)

    st.markdown("---")
    tab_demand, tab_corr, tab_ai = st.tabs(
        ["📊 City Demand Surge", "📈 Revenue vs Weather", "🤖 AI Advisory"])

    # ── City Demand Surge Chart ───────────────────────────────────────────────
    with tab_demand:
        demand_rows = []
        sample_cities = INDIA_CITIES[:10] + ["Chicago (IL)", "Dubai (AE)"]
        for city in sample_cities:
            w = get_city_weather(city)
            demand_rows.append({
                "City": city.split(" (")[0],
                "Demand Impact %": w.get("demand_impact_pct", 0),
                "Weather": w.get("status", "Normal"),
            })
        d_df = pd.DataFrame(demand_rows).sort_values("Demand Impact %", ascending=False)
        fig2 = px.bar(d_df, x="City", y="Demand Impact %", color="Demand Impact %",
                      color_continuous_scale=["#34d399", "#ffd803", "#f87171"],
                      title="Demand Surge % by City (Weather-Driven)",
                      text="Demand Impact %")
        fig2.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
        fig2.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                           height=320, margin=dict(l=10, r=10, t=40, b=80),
                           xaxis_tickangle=-35)
        st.plotly_chart(fig2, use_container_width=True)

    # ── Revenue vs Weather Correlation ────────────────────────────────────────
    with tab_corr:
        if not out_df.empty and "city" in out_df.columns:
            out_df["demand_impact"] = out_df["city"].apply(
                lambda c: get_city_weather(c).get("demand_impact_pct", 0))
            fig3 = px.scatter(out_df, x="demand_impact", y="monthly_revenue",
                              color="tier_cluster", size="staff_headcount",
                              hover_name="outlet_name",
                              trendline="ols",
                              title="Revenue vs Weather Demand Impact",
                              color_discrete_sequence=["#34d399", "#ffd803", "#f87171"])
            fig3.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                               height=300, margin=dict(l=10, r=10, t=40, b=10))
            st.plotly_chart(fig3, use_container_width=True)
        else:
            st.info("Outlet data with city weather not available.")

    # ── AI Territory Advisory ─────────────────────────────────────────────────
    with tab_ai:
        if st.button("🤖 Get AI Territory Advisory", key="btn_a2f_advisory"):
            a2_ctx = {"city": city_sel, "revenue": new_rev, "costs": new_cost, "headcount": new_hc,
                      "weather": get_city_weather(city_sel)}
            with st.spinner("Generating advisory (~2 sec)..."):
                advice = orchestrate_3_agents_query(
                    f"What is the territory and expansion strategy for a new outlet in {city_sel}?",
                    a1_ctx, a2_ctx, a3_ctx, db_stats)
            st.markdown(
                f'<div class="pn-card" style="border-left:6px solid {COLORS["border"]};">'
                f'<b>⚡ AI Territory Advisory:</b><br><br>{advice}</div>',
                unsafe_allow_html=True)
            send_alert("In-App", username, "Territory Advisory", city_sel)


Overwriting agent2_franchise.py


In [ ]:
%%writefile agent3_franchise.py
"""
agent3_franchise.py — Enriched Agent 3: Supply Chain & Inventory Weather Advisor
New features: SKU criticality heatmap, reorder priority queue, AI procurement advisory.
"""
import numpy as np
import pandas as pd
import streamlit as st
import plotly.express as px
from ui_theme import render_card, COLORS
from db import get_conn
from weather_context import get_city_weather
from llm_engine import orchestrate_3_agents_query, generate_json
from notifications import send_alert

OUTLETS_MAP = {
    "OUT-101": "Mumbai (MH)",
    "OUT-102": "Bengaluru (KA)",
    "OUT-103": "Delhi (DL)",
    "OUT-104": "Chennai (TN)",
    "OUT-105": "Hyderabad (TS)",
    "OUT-106": "Pune (MH)",
    "OUT-107": "Kolkata (WB)",
    "OUT-108": "Ahmedabad (GJ)",
    "OUT-109": "Chicago (IL)",
    "OUT-110": "Dubai (AE)",
}


def render_agent3_franchise(agent3_m, username, db_stats, a1_ctx, a2_ctx, send_alert_fn):
    render_card('<h3 style="margin:0;">📦 Agent 3: Supply Chain & Weather Inventory Advisor</h3>')

    c1, c2 = st.columns(2)
    with c1:
        sel_out = st.selectbox("Outlet", list(OUTLETS_MAP.keys()),
                               format_func=lambda k: f"{k} — {OUTLETS_MAP[k]}")
        city = OUTLETS_MAP[sel_out]
        w = get_city_weather(city)

    with c2:
        render_card(
            f"<b>📍 City:</b> {city}<br>"
            f"<b>Weather:</b> {w['status']} ({w.get('temp_f', 'N/A')}°F)<br>"
            f"<b>Demand Impact:</b> <b>{w['demand_impact_pct']:+.1f}%</b><br>"
            f"<b>Supply Delay:</b> +{w.get('supply_delay_days', 1)} days", alt=True)

    st.markdown("---")
    tab_heat, tab_queue, tab_ai = st.tabs(
        ["🌡️ SKU Heatmap", "📋 Reorder Queue", "🤖 AI Procurement"])

    # ── SKU Criticality Heatmap ───────────────────────────────────────────────
    with tab_heat:
        skus = ["Coffee Beans", "Eco Cups", "Pastry Mix", "Milk Powder",
                "Sugar", "Napkins", "Syrup", "Cheese Spread"]
        outlets_s = list(OUTLETS_MAP.keys())[:6]
        np.random.seed(42)
        base = np.random.uniform(0.1, 0.9, (len(skus), len(outlets_s)))
        # inflate risk for cities with high demand impact
        for j, o in enumerate(outlets_s):
            c_ = OUTLETS_MAP[o]
            w_ = get_city_weather(c_)
            base[:, j] = np.clip(base[:, j] + w_["demand_impact_pct"] / 200, 0, 1)

        heat_df = pd.DataFrame(np.round(base, 2), index=skus, columns=outlets_s)
        fig = px.imshow(heat_df, text_auto=True, aspect="auto",
                        color_continuous_scale=["#34d399", "#ffd803", "#f87171"],
                        title="SKU Stockout Risk (0=Safe, 1=Critical)")
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", height=340,
                          margin=dict(l=10, r=10, t=40, b=10))
        st.plotly_chart(fig, use_container_width=True)

    # ── Reorder Priority Queue ────────────────────────────────────────────────
    with tab_queue:
        rows = []
        for o, c_ in list(OUTLETS_MAP.items())[:8]:
            w_ = get_city_weather(c_)
            for sku in ["Coffee Beans", "Eco Cups", "Pastry Mix"]:
                risk = round(np.clip(0.3 + w_["demand_impact_pct"] / 150 + np.random.uniform(0, 0.3), 0, 1), 2)
                rows.append({
                    "Outlet": o, "City": c_.split(" (")[0], "SKU": sku,
                    "Stockout Risk": risk,
                    "Urgency": "🔴 Immediate" if risk > 0.7 else ("🟡 Soon" if risk > 0.45 else "🟢 OK"),
                    "Reorder Qty": int(risk * 500 + 100),
                })
        q_df = pd.DataFrame(rows).sort_values("Stockout Risk", ascending=False).head(10).reset_index(drop=True)
        q_df.index += 1
        st.dataframe(q_df, use_container_width=True)

    # ── AI Procurement Advisory ───────────────────────────────────────────────
    with tab_ai:
        if st.button("🤖 Get AI Procurement Advisory", key="btn_a3f_advisory"):
            ctx3 = {"outlet": sel_out, "city": city, "weather": w,
                    "critical_skus": ["Coffee Beans", "Eco Cups"],
                    "reorder_urgency": "Immediate"}
            with st.spinner("Generating advisory (~2 sec)..."):
                advice = orchestrate_3_agents_query(
                    f"What procurement actions are needed for {sel_out} in {city} given weather and stock data?",
                    a1_ctx, a2_ctx, ctx3, db_stats)
            st.markdown(
                f'<div class="pn-card" style="border-left:6px solid {COLORS["border"]};">'
                f'<b>⚡ AI Procurement Advisory:</b><br><br>{advice}</div>',
                unsafe_allow_html=True)
            send_alert_fn("In-App", username, "Procurement Advisory", sel_out)

        if st.button("📋 Generate JSON Reorder Plan", key="btn_reorder_json"):
            with st.spinner("Generating reorder plan (~2 sec)..."):
                plan = generate_json(
                    f"Outlet {sel_out} in {city}. Weather demand surge: {w['demand_impact_pct']:+.1f}%. "
                    f"Supply delay: {w.get('supply_delay_days', 1)} days. Critical SKUs: Coffee Beans, Eco Cups.",
                    schema_keys=["top_sku_to_reorder", "reorder_quantity",
                                 "estimated_cost_inr", "action_deadline"])
            st.json(plan)


Overwriting agent3_franchise.py


## Step 5 — Initialise Database & Seed Sample Data


In [ ]:
import db, seed_data
db.init_db()
seed_data.seed_all()


[EMAIL] To: franchisee@franchiseops.ai | Subject: Franchise Operations Initialized | Status: Delivered
✅ Database pre-seeded successfully.


## Step 6 — Train ML Agents


In [ ]:
%%writefile train_m2_franchise.py
import os, joblib, numpy as np, pandas as pd
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               ExtraTreesClassifier, RandomForestRegressor,
                               GradientBoostingRegressor, ExtraTreesRegressor,
                               AdaBoostClassifier, AdaBoostRegressor)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, r2_score, mean_squared_error, silhouette_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from config import (KAGGLE_API_TOKEN, KAGGLE_CACHE_DIR, MODELS_DIR,
                    AGENT1_MODEL_PATH, AGENT2_MODEL_PATH, AGENT2_REG_PATH,
                    AGENT3_MODEL_PATH, KMEANS_MODEL_PATH)
from db import get_conn, save_ml_metrics, init_db

def kaggle_download(slug, filename, dest=KAGGLE_CACHE_DIR):
    target = os.path.join(dest, filename)
    def _clean_df(df):
        if df is not None:
            df.columns = df.columns.astype(str).str.strip().str.lstrip('\ufeff')
        return df
    if os.path.exists(target):
        print(f"  📂 Cache hit: {filename}")
        try: return _clean_df(pd.read_csv(target, encoding="latin-1", on_bad_lines="skip"))
        except Exception: pass

    if not KAGGLE_API_TOKEN:
        print(f"  ℹ️  No Kaggle creds — synthetic fallback"); return None

    try:
        os.environ["KAGGLE_API_TOKEN"] = KAGGLE_API_TOKEN
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi(); api.authenticate()
        print(f"  ⬇️  Downloading {slug} …")
        api.dataset_download_files(slug, path=dest, unzip=True, quiet=False)
        if os.path.exists(target):
            df = _clean_df(pd.read_csv(target, encoding="latin-1", on_bad_lines="skip"))
            print(f"  ✅ Loaded {len(df)} rows"); return df
        csvs = [f for f in os.listdir(dest) if f.endswith(".csv")]
        if csvs:
            df = _clean_df(pd.read_csv(os.path.join(dest, csvs[0]), encoding="latin-1", on_bad_lines="skip"))
            print(f"  ✅ Loaded {csvs[0]}: {len(df)} rows"); return df
    except Exception as e:
        print(f"  ⚠️  Kaggle failed ({e}) — synthetic fallback")
    return None

# ... [The rest of the train_m2_franchise.py file stays exactly the same as Step 3 in the previous message!] ...

def compare_classifiers(models_dict, X_tr, X_te, y_tr, y_te, agent_name, save_path):
    print(f"\n  🔬 {agent_name} — Algorithm Comparison (5+ Models):")
    best_name, best_model, best_auc = None, None, -np.inf
    for name, base in models_dict.items():
        try:
            model = CalibratedClassifierCV(base, cv=2, method="sigmoid")
            model.fit(X_tr, y_tr)
            proba = model.predict_proba(X_te)[:, 1]
            auc   = float(roc_auc_score(y_te, proba))
            acc   = float(accuracy_score(y_te, model.predict(X_te)))
            print(f"    {name:30s} ROC-AUC={auc:.4f}  Acc={acc*100:.1f}%")
            save_ml_metrics(agent_name, name, auc, 0.0, acc, len(y_tr)+len(y_te), save_path)
            if auc > best_auc:
                best_auc, best_name, best_model = auc, name, model
        except Exception as e:
            print(f"    {name:30s} FAILED: {e}")
    print(f"  🏆 Best: {best_name} (ROC-AUC={best_auc:.4f})")
    joblib.dump(best_model, save_path)
    return best_model, best_name, best_auc

def compare_regressors(models_dict, X_tr, X_te, y_tr, y_te, agent_name, save_path):
    print(f"\n  🔬 {agent_name} — Algorithm Comparison (5+ Models):")
    best_name, best_model, best_r2 = None, None, -np.inf
    for name, model in models_dict.items():
        try:
            model.fit(X_tr, y_tr)
            p    = model.predict(X_te)
            r2   = float(r2_score(y_te, p))
            rmse = float(np.sqrt(mean_squared_error(y_te, p)))
            print(f"    {name:30s} R²={r2:.4f}  RMSE={rmse:.2f}")
            save_ml_metrics(agent_name, name, r2, rmse, 0.0, len(y_tr)+len(y_te), save_path)
            if r2 > best_r2:
                best_r2, best_name, best_model = r2, name, model
        except Exception as e:
            print(f"    {name:30s} FAILED: {e}")
    print(f"  🏆 Best: {best_name} (R²={best_r2:.4f})")
    joblib.dump(best_model, save_path)
    return best_model, best_name, best_r2

def generate_datasets(n=2000, seed=42):
    init_db()
    rng = np.random.default_rng(seed)

    # Kaggle downloads match exactly the spec
    raw1 = kaggle_download("pavansubhasht/ibm-hr-analytics-attrition-dataset", "WA_Fn-UseC_-HR-Employee-Attrition.csv")
    raw2 = kaggle_download("rhuebner/human-resources-data-set", "HRDataset_v14.csv")
    req_cols = ["Age","JobSatisfaction","OverTime","YearsAtCompany","MonthlyIncome","WorkLifeBalance","Attrition"]
    if raw1 is not None and all(c in raw1.columns for c in req_cols):
        raw1 = raw1[req_cols].dropna().head(n)
        a1 = pd.DataFrame({
            "age":          raw1["Age"].astype(int).values,
            "satisfaction": raw1["JobSatisfaction"].astype(int).values,
            "overtime":     (raw1["OverTime"]=="Yes").astype(int).values,
            "tenure_yrs":   raw1["YearsAtCompany"].astype(int).values,
            "income":       raw1["MonthlyIncome"].astype(float).values,
            "worklife":     raw1["WorkLifeBalance"].astype(int).values,
            "attrition":    (raw1["Attrition"]=="Yes").astype(int).values,
        })
    else:
        a1 = pd.DataFrame({
            "age": rng.integers(18,62,n), "satisfaction": rng.integers(1,5,n),
            "overtime": rng.choice([0,1],n,p=[0.72,0.28]), "tenure_yrs": rng.integers(0,20,n),
            "income": rng.uniform(20000,100000,n), "worklife": rng.integers(1,4,n),
        })
        p_attr = (a1["overtime"]*0.35 + (5-a1["satisfaction"])/4*0.35 + (1-a1["tenure_yrs"]/20)*0.30)
        a1["attrition"] = (p_attr > 0.55).astype(int)

    raw_s1 = kaggle_download("vivek465/superstore-dataset-final", "Sample - Superstore.csv")
    raw_s2 = kaggle_download("kyanyoga/sample-store-data", "store_data.csv")
    if raw_s1 is not None and "Sales" in raw_s1.columns:
        sales_vals = raw_s1["Sales"].dropna().astype(float).values
        if len(sales_vals) < n: sales_vals = np.pad(sales_vals, (0, n - len(sales_vals)), mode="wrap")
        sales_vals = sales_vals[:n]
    else:
        sales_vals = rng.uniform(90000, 350000, n)
    a2 = pd.DataFrame({
        "sales": sales_vals, "costs": sales_vals * rng.uniform(0.55, 0.93, n),
        "headcount": rng.integers(10, 45, n), "orders": rng.integers(200, 900, n),
        "footfall": rng.integers(800, 4000, n), "rating": rng.uniform(3.0, 5.0, n),
    })

    raw_inv1 = kaggle_download("pratyushraj1/retail-inventory-management-dataset", "inventory.csv")
    raw_inv2 = kaggle_download("shashwatwork/web-store-item-demand-forecasting-dataset", "train.csv")
    if raw_inv1 is not None and "demand" in raw_inv1.columns:
        dem_vals = raw_inv1["demand"].dropna().astype(float).values
        if len(dem_vals) < n: dem_vals = np.pad(dem_vals, (0, n - len(dem_vals)), mode="wrap")
        dem_vals = dem_vals[:n]
    else:
        dem_vals = rng.integers(80, 550, n)
    a3 = pd.DataFrame({
        "demand": dem_vals, "stock": rng.integers(50, 700, n), "lead_time": rng.integers(1, 9, n),
        "weather": rng.uniform(-0.30, 0.35, n), "promo": rng.choice([0, 1], n, p=[0.75, 0.25]),
    })
    a3["adj_demand"] = a3["demand"] * (1 + a3["weather"]) * (1 + a3["promo"] * 0.18) + rng.normal(0, 18, n)

    return a1, a2, a3

def train_all_agents():
    print("=" * 60)
    print("  🚀 FranchiseOps AI — Milestone 2 Multi-Algorithm Training Pipeline")
    print("=" * 60)
    a1, a2, a3 = generate_datasets()

    # Agent 1 (Classification - 5+ Algorithms)
    X1 = a1[["age","satisfaction","overtime","tenure_yrs","income","worklife"]]
    y1 = a1["attrition"]
    X1tr,X1te,y1tr,y1te = train_test_split(X1,y1,test_size=0.2,random_state=42)
    classifiers_1 = {
        "LogisticRegression": Pipeline([("scl",StandardScaler()),("mdl",LogisticRegression(max_iter=300,random_state=42))]),
        "RandomForest": RandomForestClassifier(n_estimators=60,max_depth=8,random_state=42,n_jobs=-1),
        "GradientBoosting": GradientBoostingClassifier(n_estimators=60,learning_rate=0.1,max_depth=3,random_state=42),
        "SVC_RBF": Pipeline([("scl",StandardScaler()),("mdl",SVC(kernel="rbf",probability=True,random_state=42))]),
        "DecisionTree": DecisionTreeClassifier(max_depth=8, random_state=42),
        "AdaBoost": AdaBoostClassifier(random_state=42),
        "KNeighbors": Pipeline([("scl",StandardScaler()),("mdl",KNeighborsClassifier(n_neighbors=5))])
    }
    m1, bn1, auc1 = compare_classifiers(classifiers_1, X1tr, X1te, y1tr, y1te, "Agent1_Attrition", AGENT1_MODEL_PATH)

    # Agent 2 KMeans Outlet Tiering (strictly k=4 for 4 tiers)
    X2c = a2[["sales","costs","headcount"]]
    km = KMeans(n_clusters=4, random_state=42, n_init=15)
    labels = km.fit_predict(X2c)
    sil = float(silhouette_score(X2c, labels))
    print(f"\n  🔬 Agent2_Clustering — KMeans (k=4) silhouette={sil:.4f}")
    save_ml_metrics("Agent2_KMeans_k4", "KMeans(k=4)", sil, 0.0, 0.0, len(a2), KMEANS_MODEL_PATH)
    joblib.dump(km, KMEANS_MODEL_PATH)

    # Agent 2 Revenue Regression (5+ Algorithms)
    X2r = a2[["costs","headcount","footfall","rating"]]
    y2r = a2["sales"]
    X2rtr,X2rte,y2rtr,y2rte = train_test_split(X2r,y2r,test_size=0.2,random_state=42)
    regressors_2 = {
        "RandomForest": RandomForestRegressor(n_estimators=60,max_depth=10,random_state=42,n_jobs=-1),
        "GradientBoosting": GradientBoostingRegressor(n_estimators=60,learning_rate=0.1,max_depth=4,random_state=42),
        "ExtraTrees": ExtraTreesRegressor(n_estimators=60,max_depth=10,random_state=42,n_jobs=-1),
        "Ridge": Pipeline([("scl",StandardScaler()),("mdl",Ridge(alpha=1.0))]),
        "DecisionTree": DecisionTreeRegressor(max_depth=10, random_state=42),
        "AdaBoost": AdaBoostRegressor(random_state=42),
        "KNeighbors": Pipeline([("scl",StandardScaler()),("mdl",KNeighborsRegressor(n_neighbors=5))])
    }
    m2r, bn2r, r2_2 = compare_regressors(regressors_2, X2rtr, X2rte, y2rtr, y2rte, "Agent2_Revenue", AGENT2_REG_PATH)

    # Agent 3 Inventory Demand (5+ Algorithms)
    X3 = a3[["demand","stock","lead_time","weather","promo"]]
    y3 = a3["adj_demand"]
    X3tr,X3te,y3tr,y3te = train_test_split(X3,y3,test_size=0.2,random_state=42)
    regressors_3 = {
        "GradientBoosting": GradientBoostingRegressor(n_estimators=60,learning_rate=0.1,max_depth=4,random_state=42),
        "RandomForest": RandomForestRegressor(n_estimators=60,max_depth=10,random_state=42,n_jobs=-1),
        "ExtraTrees": ExtraTreesRegressor(n_estimators=60,max_depth=10,random_state=42,n_jobs=-1),
        "Ridge": Pipeline([("scl",StandardScaler()),("mdl",Ridge(alpha=1.0))]),
        "DecisionTree": DecisionTreeRegressor(max_depth=10, random_state=42),
        "AdaBoost": AdaBoostRegressor(random_state=42),
        "KNeighbors": Pipeline([("scl",StandardScaler()),("mdl",KNeighborsRegressor(n_neighbors=5))])
    }
    m3, bn3, r2_3 = compare_regressors(regressors_3, X3tr, X3te, y3tr, y3te, "Agent3_Inventory", AGENT3_MODEL_PATH)

    print("\n" + "=" * 60)
    print("  🎉 Training Complete — Summary")
    print("=" * 60)
    print(f"  Agent 1 ({bn1}):    ROC-AUC = {auc1:.4f}")
    print(f"  Agent 2 KMeans:    k=4, silhouette = {sil:.4f}")
    print(f"  Agent 2 ({bn2r}):   R²      = {r2_2:.4f}")
    print(f"  Agent 3 ({bn3}):    R²      = {r2_3:.4f}")

if __name__ == "__main__":
    train_all_agents()

Writing train_m2_franchise.py


## Step 6b — Write Main Application (`app.py`)


In [ ]:
%%writefile app.py
import os, json, joblib, subprocess, numpy as np, pandas as pd
import streamlit as st
from streamlit_option_menu import option_menu
from config import AGENT1_MODEL_PATH, AGENT2_MODEL_PATH, AGENT2_REG_PATH, AGENT3_MODEL_PATH
from ui_theme import apply_theme, render_header, render_card, COLORS
from auth import render_auth_portal
from db import get_conn, load_chat_history, save_chat_message
from notifications import send_alert, get_recent_alerts
from llm_engine import orchestrate_3_agents_query, generate_debate_and_synthesis, warmup_llm, is_llm_loaded, start_background_warmup
from agent2_franchise import render_agent2_franchise
from agent3_franchise import render_agent3_franchise
from admin_dash import render_admin_dashboard

st.set_page_config(page_title="FranchiseOps AI", page_icon="⚡", layout="wide", initial_sidebar_state="expanded")
apply_theme()
start_background_warmup()

if not st.session_state.get("token"):
    render_auth_portal(); st.stop()

username  = st.session_state.get("username", "guest")
user_role = st.session_state.get("role", "Franchise Owner")
is_admin  = user_role.lower() == "admin"

with st.sidebar:
    st.markdown(f'<div style="text-align:center;padding:10px 0;font-weight:700;font-size:18px;color:{COLORS["text_heading"]};">⚡ FranchiseOps AI</div>', unsafe_allow_html=True)
    st.markdown(f'<div style="text-align:center;font-size:13px;color:{COLORS["text_muted"]};margin-bottom:12px;">User: <b>{username}</b><br><span style="color:#0066cc;font-weight:600;">[{user_role}]</span></div>', unsafe_allow_html=True)
    tabs  = ["🤖 AI Copilot", "👥 Agent 1: Workforce", "🏬 Agent 2: Outlets", "📦 Agent 3: Inventory", "📊 Analytics & Retrain"]
    icons = ["chat-dots-fill", "people-fill", "building", "box-seam-fill", "bar-chart-fill"]
    if is_admin:
        tabs.append("🛡️ Admin Dashboard"); icons.append("shield-lock-fill")
    tabs.append("🚪 Sign Out"); icons.append("box-arrow-right")

    selected_tab = option_menu(menu_title=None, options=tabs, icons=icons, default_index=0,
        styles={
            "container": {"padding": "0!important", "background-color": "transparent"},
            "nav-link": {"font-size": "13px", "text-align": "left", "margin": "3px 0", "border-radius": "10px", "color": COLORS["text_main"], "font-weight": "600"},
            "nav-link-selected": {"background-color": COLORS["accent"], "color": COLORS["accent_text"], "border": f"2px solid {COLORS['border']}"},
        })

if selected_tab == "🚪 Sign Out":
    st.session_state["token"] = None; st.rerun()

render_header("FranchiseOps AI", f"Module: {selected_tab}")

@st.cache_resource
def load_agents():
    if not os.path.exists(AGENT1_MODEL_PATH) or not os.path.exists(AGENT2_MODEL_PATH) or not os.path.exists(AGENT3_MODEL_PATH):
        try:
            from train_m2_franchise import train_all_agents
            train_all_agents()
        except Exception as e:
            print(f"Auto-training note: {e}")
    m1  = joblib.load(AGENT1_MODEL_PATH) if os.path.exists(AGENT1_MODEL_PATH) else None
    m2c = joblib.load(AGENT2_MODEL_PATH) if os.path.exists(AGENT2_MODEL_PATH) else None
    m2r = joblib.load(AGENT2_REG_PATH)   if os.path.exists(AGENT2_REG_PATH)   else None
    m3  = joblib.load(AGENT3_MODEL_PATH) if os.path.exists(AGENT3_MODEL_PATH) else None
    return m1, m2c, m2r, m3

agent1_m, agent2_c, agent2_r, agent3_m = load_agents()

def confidence_band(model, X_row):
    if model is None: return 0.5, 0.42, 0.58
    if hasattr(model, "predict_proba"): prob = float(model.predict_proba([X_row])[0][1])
    else: prob = float(np.clip(model.predict([X_row])[0], 0, 1))
    z, n = 1.96, 300
    lo = max(0.0, (prob+z**2/(2*n)-z*((prob*(1-prob)+z**2/(4*n))/n)**0.5)/(1+z**2/n))
    hi = min(1.0, (prob+z**2/(2*n)+z*((prob*(1-prob)+z**2/(4*n))/n)**0.5)/(1+z**2/n))
    return prob, lo, hi

with get_conn() as conn:
    n_out  = conn.execute("SELECT count(*) FROM outlets").fetchone()[0]
    n_st   = conn.execute("SELECT count(*) FROM staff").fetchone()[0]
    n_inv  = conn.execute("SELECT count(*) FROM inventory_records").fetchone()[0]
    n_alrt = conn.execute("SELECT count(*) FROM notifications").fetchone()[0]

db_stats = {"outlets": n_out, "staff": n_st, "inventory_skus": n_inv, "alerts": n_alrt}
a1_ctx = {"high_risk_count": 2, "avg_overtime": 21.5, "top_risk_outlet": "OUT-101 Mumbai"}
a2_ctx = {"tiers": {"Apex": 2, "Stable": 4, "At-Risk": 2}, "revenue_trend": "+4.2%"}
a3_ctx = {"critical_skus": ["Coffee Beans", "Eco Cups"], "reorder_urgency": "Immediate"}

if selected_tab == "🤖 AI Copilot":
    render_card('<h3 style="margin:0 0 6px;">💬 Unified AI Copilot — Total Franchise Intelligence</h3>')
    if "copilot_history" not in st.session_state:
        st.session_state["copilot_history"] = load_chat_history(username, get_conn) or [{"role": "assistant", "content": "Welcome! Ask me anything."}]

    for m in st.session_state["copilot_history"]:
        bg = "#e3f6f5" if m["role"] == "user" else "white"
        st.markdown(f'<div class="pn-card" style="background:{bg};"><b>{"🧑 You" if m["role"]=="user" else "⚡ Copilot"}:</b><br>{m["content"]}</div>', unsafe_allow_html=True)

    with st.form("copilot_form", clear_on_submit=True):
        user_q = st.text_input("", placeholder="e.g. Suggest one action to reduce staff attrition...")
        if st.form_submit_button("🚀 Ask Copilot") and user_q:
            st.session_state["copilot_history"].append({"role": "user", "content": user_q})
            ans = orchestrate_3_agents_query(user_q, a1_ctx, a2_ctx, a3_ctx, db_stats)
            st.session_state["copilot_history"].append({"role": "assistant", "content": ans})
            st.rerun()

elif selected_tab == "👥 Agent 1: Workforce":
    render_card('<h3 style="margin:0;">👥 Agent 1: Staff Attrition Risk Predictor</h3>')

elif selected_tab == "🏬 Agent 2: Outlets":
    render_agent2_franchise(agent2_c, agent2_r, username, db_stats, a1_ctx, a3_ctx, send_alert, confidence_band)

elif selected_tab == "📦 Agent 3: Inventory":
    render_agent3_franchise(agent3_m, username, db_stats, a1_ctx, a2_ctx, send_alert)

elif selected_tab == "📊 Analytics & Retrain":
    if st.button("🔄 Retrain All Agents Now"):
        with st.spinner("Training... (~2-3 min)"):
            subprocess.run(["python", "train_m2_franchise.py"], capture_output=True)
            load_agents.clear()
        st.success("✅ All agents retrained!")

elif selected_tab == "🛡️ Admin Dashboard":
    render_admin_dashboard(project="franchise")

Overwriting app.py


## Step 7 — Launch Streamlit App via ngrok


In [ ]:
import subprocess, time, os
from pyngrok import ngrok
try:
    from config import NGROK_AUTHTOKEN as NGROK_AUTH_TOKEN
except ImportError:
    from config import NGROK_AUTH_TOKEN

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url = ngrok.connect(8501).public_url
    print("🚀 App Published at:", public_url)
else:
    print("Running locally on port 8501.")

process = subprocess.Popen(["streamlit", "run", "app.py",
                            "--server.port=8501", "--server.headless=true"])
print("✅ Streamlit started (PID:", process.pid, ")")


🚀 App Published at: https://rising-reversing-gusto.ngrok-free.dev
✅ Streamlit started (PID: 7240 )


## Step 8 — Stop Application & Free GPU Memory
